# Детекция меланомы | Часть 1: Борьба с дисбалансом классов

| | |
|---|---|
| **Датасет** | ISIC 2020 (33 126 изображений, менее 2% меланом) |
| **Задача** | Бинарная классификация: меланома / доброкачественное |
| **Архитектура** | ResNet-50 pretrained ImageNet, full fine-tuning |
| **Главная метрика** | AUC-PR (Average Precision) |
| **Вспомогательные** | AUC-ROC, F1, Sensitivity, Specificity, Balanced Accuracy |

---

## Логика экспериментов

Все эксперименты устроены по принципу **контролируемого изменения одного фактора** - фиксирую всё, кроме того, что изучаю. Аугментация считается частью препроцессинга и фиксируется до выбора стратегии балансировки, чтобы стратегии сравнивались в одинаковых условиях.

| Эксперимент | Что фиксируется | Что варьируется |
|---|---|---|
| **1. Сплит** | нет аугментации, обычный батч, BCE | размер val/test (10%, 12.5%, 15%) |
| **2. Аугментация** | лучший сплит, обычный батч, BCE | 5 стратегий аугментации |
| **3. Балансировка** | лучший сплит + лучшая аугментация | 5 стратегий борьбы с дисбалансом |


## 1. Конфигурация и импорты

In [1]:
import os
import copy
import json
import base64
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, HTML

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, f1_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # не засоряю логи лишним

In [ ]:
# Пути к датасетам на Kaggle
DATA_DIR  = "/kaggle/input/competitions/siim-isic-melanoma-classification"
IMAGE_DIR = "/kaggle/input/datasets/cdeotte/jpeg-melanoma-256x256/train"
CSV_PATH  = os.path.join(DATA_DIR, "train.csv")

# Гиперпараметры - всё в одном месте для воспроизводимости
IMAGE_SIZE  = 224   # стандартный входной размер для ResNet/ViT/Swin
BATCH_SIZE  = 32    # размер батча
NUM_EPOCHS  = 10    # число эпох обучения
RANDOM_SEED = 42    # фиксирую случайность

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")

Устройство: cuda


## 2. Загрузка данных

Датасет ISIC 2020. Целевой столбец `target`: 0 = доброкачественное, 1 = меланома.


In [3]:
df = pd.read_csv(CSV_PATH)
print(df.head())
print(f"Всего записей: {len(df)}")

     image_name  patient_id     sex  age_approx anatom_site_general_challenge  \
0  ISIC_2637011  IP_7279968    male        45.0                     head/neck   
1  ISIC_0015719  IP_3075186  female        45.0               upper extremity   
2  ISIC_0052212  IP_2842074  female        50.0               lower extremity   
3  ISIC_0068279  IP_6890425  female        45.0                     head/neck   
4  ISIC_0074268  IP_8723313  female        55.0               upper extremity   

  diagnosis benign_malignant  target  
0   unknown           benign       0  
1   unknown           benign       0  
2     nevus           benign       0  
3   unknown           benign       0  
4   unknown           benign       0  
Всего записей: 33126


In [4]:
counts = df["target"].value_counts()
total  = len(df)
print(f"Доля меланом: {counts[1] / total:.2%} ({counts[1]} из {total})")
print(f"Соотношение классов: 1 : {counts[0] // counts[1]}")

Доля меланом: 1.76% (584 из 33126)
Соотношение классов: 1 : 55


Вывод: менее 2% - это меланомы, то есть Accuracy бесполезна, так как модель, всегда предсказывающая 0, даёт свыше 98% точности. Главная метрика - AUC-PR: честна при дисбалансе, не зависит от порога

## 3. Препроцессинг и класс датасета


In [5]:
# Базовый препроцессинг БЕЗ аугментации.
base_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),       # меньшую сторону сжимаем до 224 пикселей
    transforms.CenterCrop(IMAGE_SIZE),   # берём центральный квадрат 224×224
    transforms.ToTensor(),               # получаем тензор с дробными числами
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],      # ResNet обучался на ImageNet, где все пиксели были отмасштабированы определённым образом
        std=[0.229, 0.224, 0.225]        # Чтобы веса модели корректно работали с новыми изображениям, их необходимо соответствующе нормализовать
    ),
])

In [ ]:
class MelanomaDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["image_name"] + ".jpg")
        image    = Image.open(img_path).convert("RGB")  # защита от RGBA/grayscale
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row["target"], dtype=torch.float32)  # float32 для BCEWithLogitsLoss
        return image, label

# Быстрая проверка
_ds = MelanomaDataset(df.head(5), IMAGE_DIR, transform=base_transforms)
img, lbl = _ds[0]
print(f"Форма тензора: {img.shape}")
print(f"Диапазон значений: [{img.min():.2f}, {img.max():.2f}]")
del _ds

Форма тензора: torch.Size([3, 224, 224])
Диапазон значений: [-1.06, 2.45]


## 4. Вспомогательные функции

In [7]:
def make_split_loaders(df, test_size, train_transform=None, random_state=RANDOM_SEED):
    """
    Разбивает df на train/val/test. test_size=0.10 -> 10% test, 10% val, 80% train.
    Стратификация по target: доля меланом одинакова в каждой части.
    Возвращает: regular_loader, balanced_loader, val_loader, test_loader.
    """
    if train_transform is None:
        train_transform = base_transforms

    df_tv, df_test = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df["target"]
    )
    val_size = test_size / (1 - test_size)
    df_train, df_val = train_test_split(
        df_tv, test_size=val_size, random_state=random_state, stratify=df_tv["target"]
    )

    total = len(df)
    print(f"Сплит {int(test_size*100)}/{int(test_size*100)}: "
          f"train {len(df_train)} ({len(df_train)/total:.1%}), "
          f"val {len(df_val)} ({len(df_val)/total:.1%}), "
          f"test {len(df_test)} ({len(df_test)/total:.1%})")

    train_ds = MelanomaDataset(df_train, IMAGE_DIR, transform=train_transform)
    val_ds   = MelanomaDataset(df_val,   IMAGE_DIR, transform=base_transforms)  # val всегда без аугм.
    test_ds  = MelanomaDataset(df_test,  IMAGE_DIR, transform=base_transforms)  # test всегда без аугм.

    # Обычный загрузчик: случайный порядок батчей
    regular_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=4, persistent_workers=True, pin_memory=True)

    # Сбалансированный загрузчик: меланомы и здоровые тянутся с равной вероятностью
    labels = train_ds.df["target"].values
    n_neg, n_pos = (labels == 0).sum(), (labels == 1).sum()
    sample_weights = torch.tensor(
        [1.0/n_pos if l == 1 else 1.0/n_neg for l in labels], dtype=torch.float32
    )
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    balanced_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                 shuffle=False, num_workers=4,
                                 persistent_workers=True, pin_memory=True)

    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, persistent_workers=True, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, persistent_workers=True, pin_memory=True)

    return regular_loader, balanced_loader, val_loader, test_loader, df_train

In [8]:
def make_resnet50():
    """
    ResNet-50 pretrained ImageNet. Последний слой заменяем: 1000 классов -> 1 нейрон.
    Один нейрон = logit, BCEWithLogitsLoss сам применяет сигмоиду внутри.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(DEVICE)

## 5. Функции обучения и оценки

**Ключевое методологическое решение про порог (threshold):**  
Порог - это гиперпараметр, так что подбирать его на тестовых данных - это data leakage.  
Ищу оптимальный порог на **валидации** и применяем **тот же** на тесте.


In [11]:
def train_one_epoch(model, loader, optimizer, criterion):
    """Одна эпоха обучения. Возвращает средний лосс за эпоху."""
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)  # [B] -> [B,1] для BCEWithLogitsLoss
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [12]:
def compute_metrics(model, loader, threshold=None):
    """
    Считает все метрики модели на данных.

    threshold=None  -> ищет оптимальный порог по F1 на валидации
    threshold=число -> применяет этот порог без поиска на тесте
    """
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(model(images.to(DEVICE))).squeeze(1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    
    # Защита от NaN: если модель выдала нечисловые значения - trial провальный
    if np.isnan(all_probs).any() or np.isinf(all_probs).any():
        all_probs = np.nan_to_num(all_probs, nan=0.0, posinf=1.0, neginf=0.0)

    # AUC-PR: площадь под Precision-Recall кривой.
    # Главная метрика при дисбалансе: не зависит от порога, штрафует за каждую пропущенную меланому. 
    # AUC-ROC при дисбалансе обманчив - модель, всегда предсказывающая 0, даёт высокий AUC-ROC.
    auc_pr = average_precision_score(all_labels, all_probs)

    # Подбор оптимального порога по F1
    # Порог - граница решения: если вероятность >= порога -> предсказываем меланому. По умолчанию порог = 0,5, но при дисбалансе лучший порог часто намного ниже.
    # Перебираю все пороги от 0,01 до 0,99 с шагом 0,01, беру лучший по F1.
    if threshold is None:
        best_f1, best_thr = 0.0, 0.5
        for thr in np.arange(0.01, 0.99, 0.01):
            preds = (all_probs >= thr).astype(int)
            f1 = f1_score(all_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, float(thr)
    else:
        best_thr = float(threshold)
        preds    = (all_probs >= best_thr).astype(int)
        best_f1  = f1_score(all_labels, preds, zero_division=0)

    preds = (all_probs >= best_thr).astype(int)

    # Матрица ошибок — четыре возможных исхода:
    # TP (True Positive):  предсказали меланому — она есть  -> правильно, поймали
    # TN (True Negative):  предсказали норму    — она есть  -> правильно, не напугали
    # FP (False Positive): предсказали меланому — её нет    -> ложная тревога
    # FN (False Negative): предсказали норму    — меланома  -> пропустили болезнь
    tp = int(((preds == 1) & (all_labels == 1)).sum())
    tn = int(((preds == 0) & (all_labels == 0)).sum())
    fp = int(((preds == 1) & (all_labels == 0)).sum())
    fn = int(((preds == 0) & (all_labels == 1)).sum())

    # Recall (или же Sensitivity в медицине) = TP / (TP + FN)
    # "Из всех реальных меланом — сколько мы поймали?"
    # Самый важный вопрос медицины: пропущенная меланома опаснее ложной тревоги.
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Precision = TP / (TP + FP)
    # "Из всех кого мы назвали меланомой - сколько ею реально оказались?"
    # Важна для пациента: не хочется зря пугаться.
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # Specificity = TN / (TN + FP)
    # "Из всех здоровых - скольких мы правильно назвали здоровыми?"
    # Проверяю и метрику на доброкачественных образованиях
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # F1 = (2 * Precision * Recall) / (Precision + Recall)
    # Среднее гармоническое Precision и Recall.
    # Хорош когда нужен баланс: не только ловить меланомы, но и не кричать зря.
    # Уже посчитан выше при подборе порога, переиспользуем.

    # Balanced Accuracy = (Recall + Specificity) / 2
    # Среднее качество на обоих классах одновременно.
    balanced_acc = (recall + specificity) / 2.0

    return {
        "auc_pr":       auc_pr,
        "f1":           best_f1,
        "recall":       recall,       # = Sensitivity: сколько меланом поймали
        "precision":    precision,    # из предсказанных меланом — сколько верных
        "specificity":  specificity,  # сколько здоровых правильно определили
        "balanced_acc": balanced_acc, # честный средний балл по обоим классам
        "threshold":    best_thr,     # порог, при котором посчитаны пороговые метрики
    }

In [13]:
def run_experiment(name, train_loader, val_loader, test_loader, criterion=None):
    """
    Полный цикл обучения + оценки одного эксперимента.
    1. Одинаковый старт для всех моделей
    2. Обучаем NUM_EPOCHS эпох, лучшая эпоха выбирается по валидационному AUC-PR
    3. Сохраняется валидационный порог с лучшей эпохи — применяется на тесте
    """
    if criterion is None:
        criterion = nn.BCEWithLogitsLoss()

    print()
    print(f"--- Обучение: {name} ---")
    model     = make_resnet50()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_val_auc_pr    = 0.0
    best_val_threshold = 0.5
    best_model_wts     = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss  = train_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = compute_metrics(model, val_loader, threshold=None)

        is_best = val_metrics["auc_pr"] > best_val_auc_pr
        print(
            f"  Эп {epoch:02d}/{NUM_EPOCHS} | loss {train_loss:.4f} | "
            f"val AUC-PR {val_metrics['auc_pr']:.4f} | F1 {val_metrics['f1']:.4f}"
            + (" <- лучший" if is_best else ""), flush=True
        )
        if is_best:
            best_val_auc_pr    = val_metrics["auc_pr"]
            best_val_threshold = val_metrics["threshold"]
            best_model_wts     = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)

    test_m = compute_metrics(model, test_loader, threshold=best_val_threshold)
    print(
        f"  TEST: AUC-PR {test_m['auc_pr']:.4f} | "
        f"F1 {test_m['f1']:.4f} (thr={test_m['threshold']:.2f}) | "
        f"Recall {test_m['recall']:.4f} | Precision {test_m['precision']:.4f} | "
        f"Spec {test_m['specificity']:.4f} | BalAcc {test_m['balanced_acc']:.4f}"
    )

    return {
        "name":          name,
        "val_auc_pr":    best_val_auc_pr,
        "test_auc_pr":   test_m["auc_pr"],
        "test_f1":       test_m["f1"],
        "test_recall":   test_m["recall"],
        "test_precision":test_m["precision"],
        "test_spec":     test_m["specificity"],
        "test_bal_acc":  test_m["balanced_acc"],
        "test_thr":      test_m["threshold"],
        "model":         model,
    }

In [14]:
def print_results_table(results, title):
    """Таблица результатов экспериментов."""
    W = 112
    print()
    print("=" * W)
    print(f"  {title}")
    print("=" * W)
    print(f"{'Эксперимент':<38} {'AUC-PR':>8} {'F1':>7} "
          f"{'Recall':>8} {'Precision':>10} {'Spec':>7} {'BalAcc':>8}")
    print("-" * W)
    for r in results:
        print(f"{r['name']:<38} {r['test_auc_pr']:>8.4f} {r['test_f1']:>7.4f} "
              f"{r['test_recall']:>8.4f} {r['test_precision']:>10.4f} "
              f"{r['test_spec']:>7.4f} {r['test_bal_acc']:>8.4f}")
    best = max(results, key=lambda x: x["test_auc_pr"])
    print()
    print(f"Лучший по AUC-PR: {best['name']} -> {best['test_auc_pr']:.4f}")
    return best

## 6. Эксперимент 1: выбор сплита

**Что делаем:** перебираем три варианта разбиения данных.

**Что фиксируем:** обычный батч + BCE + без аугментации - нет ничего лишнего, чистый baseline.

**Что варьируем:** доля val/test (10%, 12.5%, 15%).

Слишком маленький test -> мало меланом (58) -> нестабильная оценка AUC-PR.  
Слишком большой test -> мало train -> модель хуже обучается.  
Задача: найти баланс.


In [16]:
split_results = []

for split_size in [0.10, 0.125, 0.15]:
    regular_loader, _, val_loader, test_loader, _ = make_split_loaders(df, split_size)
    result = run_experiment(
        name=f"Сплит {int(split_size*100)}/{int(split_size*100)}",
        train_loader=regular_loader,
        val_loader=val_loader,
        test_loader=test_loader
    )
    result["split_size"] = split_size
    split_results.append(result)

best_split = print_results_table(split_results, "Эксперимент 1: выбор сплита")
best_size  = best_split["split_size"]
print(f"Лучший сплит: {int(best_size*100)}/{int(best_size*100)} - использую далее во всех экспериментах.")

Сплит 10/10: train 26500 (80.0%), val 3313 (10.0%), test 3313 (10.0%)

--- Обучение: Сплит 10/10 ---
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 218MB/s]


  Эп 01/10 | loss 0.0831 | val AUC-PR 0.0879 | F1 0.1239 <- лучший
  Эп 02/10 | loss 0.0731 | val AUC-PR 0.1206 | F1 0.1878 <- лучший
  Эп 03/10 | loss 0.0681 | val AUC-PR 0.1364 | F1 0.2517 <- лучший
  Эп 04/10 | loss 0.0639 | val AUC-PR 0.1162 | F1 0.2029
  Эп 05/10 | loss 0.0567 | val AUC-PR 0.1792 | F1 0.2553 <- лучший
  Эп 06/10 | loss 0.0430 | val AUC-PR 0.0843 | F1 0.1545
  Эп 07/10 | loss 0.0267 | val AUC-PR 0.1012 | F1 0.2184
  Эп 08/10 | loss 0.0127 | val AUC-PR 0.1134 | F1 0.1867
  Эп 09/10 | loss 0.0114 | val AUC-PR 0.0959 | F1 0.1690
  Эп 10/10 | loss 0.0064 | val AUC-PR 0.1003 | F1 0.1325
  TEST: AUC-PR 0.2204 | F1 0.2500 (thr=0.17) | Recall 0.2414 | Precision 0.2593 | Spec 0.9877 | BalAcc 0.6145
Сплит 12/12: train 24844 (75.0%), val 4141 (12.5%), test 4141 (12.5%)

--- Обучение: Сплит 12/12 ---
  Эп 01/10 | loss 0.0852 | val AUC-PR 0.1333 | F1 0.2083 <- лучший
  Эп 02/10 | loss 0.0713 | val AUC-PR 0.1460 | F1 0.2205 <- лучший
  Эп 03/10 | loss 0.0676 | val AUC-PR 0.1645 

### Вывод по Эксперименту 1

Лучшим по основной метрике AUC-PR оказался **сплит 10/10 (0,2204)** — 80% данных на обучение, по 10% на валидацию и тест. Он фиксируется для всех последующих экспериментов.

Сплиты 12/12 и 15/15 показали заметно худший AUC-PR (0,12–0,13), что объясняется уменьшением обучающей выборки: при сильном дисбалансе (1,76% меланом) каждый дополнительный процент данных в train критически важен, потому что меланом и так катастрофически мало.

Все три сплита дают высокую Specificity (>0,98) и низкий Recall (<0,25), то есть модель почти не видит меланомы и предсказывает преимущественно здоровый класс. Это ожидаемое поведение без каких-либо стратегий борьбы с дисбалансом, то есть бейслайн честно показывает масштаб проблемы, которую предстоит решить.

## 7. Эксперимент 2: выбор стратегии аугментации

**Почему аугментация до стратегий балансировки?**  
В литературе по медицинскому CV и в пайплайнах ISIC аугментация рассматривается как **препроцессинг**, то есть часть подготовки данных, которая фиксируется до выбора стратегии обучения. Это позволяет затем сравнивать стратегии балансировки в одинаковых условиях.

**Что фиксируется:** лучший сплит + обычный батч + BCE.

**Что варьируется:** 5 стратегий аугментации для train.

Сравниваем с базой: лучший сплит + BCE + **без аугментации**.

In [15]:
# Определяем 5 стратегий аугментации
augmentation_strategies = {

    "Базовая геометрическая аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),   # случайный вырез вместо центрального
        transforms.RandomHorizontalFlip(),   # отражение по горизонтали (p=0,5)
        transforms.RandomVerticalFlip(),     # отражение по вертикали (p=0,5)
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Расширенная геометрическая аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),  # поворот в диапазоне 45 градусов
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Цветовая аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        # ColorJitter - случайные вариации яркости/контраста/насыщенности
        # Важно для дерматоскопии: освещение и аппарат влияют на цвет снимка
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Комбинированная аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Агрессивная аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        # RandomErasing: закрашивает случайный прямоугольник
        # Симулирует артефакты дерматоскопии: волосы, блики, пузырьки геля
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
    ]),
}

In [18]:
# Базовая точка: лучший сплит + без аугментации + BCE (результат из эксперимента 1).
aug_results = [{
    "name":           f"Без аугментации (Сплит {int(best_size*100)}/{int(best_size*100)})",
    "test_auc_pr":    best_split["test_auc_pr"],
    "test_f1":        best_split["test_f1"],
    "test_recall":    best_split["test_recall"],
    "test_precision": best_split["test_precision"],
    "test_spec":      best_split["test_spec"],
    "test_bal_acc":   best_split["test_bal_acc"],
    "test_thr":       best_split["test_thr"],
}]

# Запускаю 5 аугментаций с обычным батчем и BCE
for aug_name, aug_transform in augmentation_strategies.items():
    regular_loader, _, val_loader, test_loader, _ = make_split_loaders(
        df, best_size, train_transform=aug_transform
    )
    result = run_experiment(
        name=aug_name,
        train_loader=regular_loader,
        val_loader=val_loader,
        test_loader=test_loader
    )
    aug_results.append(result)

best_aug      = print_results_table(aug_results, "Эксперимент 2: выбор аугментации")
best_aug_name = best_aug["name"]

# Если лучшей оказалась базовая точка без аугментации - берём None, make_split_loaders при train_transform=None сам подставит base_transforms
best_aug_transform = augmentation_strategies.get(best_aug_name, None)
print(f"Лучшая аугментация: {best_aug_name}.")

Сплит 10/10: train 26500 (80.0%), val 3313 (10.0%), test 3313 (10.0%)

--- Обучение: Базовая геометрическая аугментация ---
  Эп 01/10 | loss 0.0850 | val AUC-PR 0.0550 | F1 0.1145 <- лучший
  Эп 02/10 | loss 0.0729 | val AUC-PR 0.1163 | F1 0.2119 <- лучший
  Эп 03/10 | loss 0.0711 | val AUC-PR 0.1245 | F1 0.2058 <- лучший
  Эп 04/10 | loss 0.0695 | val AUC-PR 0.1034 | F1 0.2500
  Эп 05/10 | loss 0.0691 | val AUC-PR 0.1448 | F1 0.2203 <- лучший
  Эп 06/10 | loss 0.0685 | val AUC-PR 0.1072 | F1 0.1980
  Эп 07/10 | loss 0.0659 | val AUC-PR 0.1323 | F1 0.1964
  Эп 08/10 | loss 0.0664 | val AUC-PR 0.1253 | F1 0.2011
  Эп 09/10 | loss 0.0645 | val AUC-PR 0.1170 | F1 0.2128
  Эп 10/10 | loss 0.0634 | val AUC-PR 0.1416 | F1 0.2320
  TEST: AUC-PR 0.2266 | F1 0.2241 (thr=0.12) | Recall 0.4483 | Precision 0.1494 | Spec 0.9545 | BalAcc 0.7014
Сплит 10/10: train 26500 (80.0%), val 3313 (10.0%), test 3313 (10.0%)

--- Обучение: Расширенная геометрическая аугментация ---
  Эп 01/10 | loss 0.0843 | v

### Вывод по Эксперименту 2

Лучшей по основной метрике AUC-PR оказалась **базовая геометрическая аугментация (0,2266)** - случайный кроп и отражения по горизонтали и вертикали. Она фиксируется для эксперимента 3.

Хоть и **агрессивная аугментация** показала наилучший Recall (0,5345) и Balanced Accuracy (0,7427), однако при этом просела по Precision (0,1623), то есть модель стала агрессивно предсказывать меланому, порождая много ложных тревог, что опасно в медицине. AUC-PR это штрафует, поскольку учитывает баланс между Precision и Recall по всей кривой.

Общий уровень AUC-PR невысок (0,12-0,23), что ожидаемо на данном этапе: стратегии борьбы с дисбалансом ещё не применялись, и модель слабо улавливает редкий класс.

## 8. Эксперимент 3: стратегии борьбы с дисбалансом классов

**Что фиксируется:** лучший сплит + лучшая аугментация.  
**Что варьируется:** 5 стратегий борьбы с дисбалансом.

Базовая точка сравнения - результат эксперимент 2 (лучший сплит + лучшая аугментация + обычный батч + BCE). Это честный baseline: аугментация та же, меняется только стратегию балансировки.

| Стратегия | Идея | Риск |
|---|---|---|
| **Оверсэмплинг** | физически дублируются меланомы 1:1 | переобучение под одни примеры |
| **Weighted BCE** | штраф за ошибку на меланоме х55 | гиперпараметр pos_weight нужно считать |
| **Balanced batch** | сэмплер вытаскивает классы поровну | меланомы всё равно повторяются |
| **Focal Loss** | автоподавление лёгких примеров | чувствителен к гиперпараметрам (alpha/gamma) |
| **Balanced batch + Focal Loss** | двойная атака на дисбаланс | сложнее отлаживать |


In [ ]:
def oversample_train(df_train, random_state=RANDOM_SEED):
    """
    Физически дублирует меланомы до соотношения 1:1 со здоровыми.
    Риск: одни и те же 584 меланомы видятся примерно 50 раз за эпоху -> возможно переобучение.
    """
    df_neg    = df_train[df_train["target"] == 0]
    df_pos    = df_train[df_train["target"] == 1]
    df_pos_up = df_pos.sample(len(df_neg), replace=True, random_state=random_state)
    df_out = (
        pd.concat([df_neg, df_pos_up])
        .sample(frac=1, random_state=random_state)  # перемешиваю - меланомы не одним блоком
        .reset_index(drop=True)
    )
    print(f"Оверсэмплинг: {len(df_neg)} здоровых + {len(df_pos_up)} меланом = {len(df_out)}")
    return df_out

In [16]:
# Сохраняю результаты сюда, чтобы при сбросе сессии в Kaggle не потерять результаты
RESULTS_PATH = "/kaggle/working/exp3_results.json"

In [ ]:
# Текущая лучшая модель
BEST_SIZE = 0.10   # лучший сплит из эксперимента 1
BEST_AUG_NAME = "Базовая геометрическая аугментация"  # лучшая аугментация из эксперимента 2

In [18]:
best_aug_transform = augmentation_strategies.get(BEST_AUG_NAME, None)
print(f"Используем аугментацию: {BEST_AUG_NAME}")
print(f"Используем сплит: {int(BEST_SIZE*100)}/{int(BEST_SIZE*100)}")

Используем аугментацию: Базовая геометрическая аугментация
Используем сплит: 10/10


In [19]:
# Метрики базовой точки (лучшая аугментация + обычный батч + BCE из Эксп. 2)
BASELINE_METRICS = {
    "name":           "Baseline (лучшая аугм. + обычный батч + BCE)",
    "test_auc_pr":    0.2266,
    "test_f1":        0.2241,
    "test_recall":    0.4483,
    "test_precision": 0.1494,
    "test_spec":      0.9545,
    "test_bal_acc":   0.7014,
    "test_thr":       0.50,
}

In [20]:
# Optuna: параметры поиска
# Диапазоны alpha и gamma стандартны для Focal Loss в медицинском CV.
N_TRIALS      = 10
ALPHA_RANGE   = (0.1, 0.5)   # вес для меланом
GAMMA_RANGE   = (0.5, 5.0)   # сила подавления лёгких примеров
OPTUNA_EPOCHS = 5            # эпох на один trial во время поиска (экономия GPU), финальное обучение всё равно NUM_EPOCHS=10 эпох
PRUNER_WARMUP = 3            # не прунить первые 3 эпохи - модели нужно время разогреться

In [21]:
# Функции для сохранения результатов
def load_results():
    """
    Загружает ранее сохранённые результаты из файла. Если файла не существует — возвращает пустой список.
    """
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Загружено {len(data)} сохранённых результатов из {RESULTS_PATH}")
        return data
    print("Файл результатов не найден.")
    return []


def save_results(results):
    """
    Сохраняет все результаты в JSON и выводит кликабельную ссылку для скачивания.
    """
    serializable = [{k: v for k, v in r.items() if k != "model"} for r in results]
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)

    # Создаём inline ссылку для скачивания прямо из ячейки (Kaggle любит не сохранять результаты)
    with open(RESULTS_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    filename = os.path.basename(RESULTS_PATH)
    href = (
        f'<a href="data:application/json;base64,{b64}" download="{filename}" '
        f'style="font-size:14px; font-weight:bold;">'
        f'Скачать {filename} ({len(serializable)} стратегий)</a>'
    )
    display(HTML(href))
    print(f"Сохранено в {RESULTS_PATH}")


def result_exists(results, name):
    """Проверяет, есть ли уже результат с таким именем, чтобы не перезапускать."""
    return any(r["name"] == name for r in results)


def get_result(results, name):
    """Возвращает результат по имени."""
    return next(r for r in results if r["name"] == name)

In [22]:
# Загрузчики с лучшей аугментацией
regular_loader, balanced_loader, val_loader, test_loader, df_train_s = make_split_loaders(
    df, BEST_SIZE, train_transform=best_aug_transform
)

# pos_weight для Weighted BCE: кол-во здоровых / кол-во меланом = 55
# Смысл: ошибка на одной меланоме = ошибка на 55 здоровых
n_neg      = (df_train_s["target"] == 0).sum()
n_pos      = (df_train_s["target"] == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"pos_weight = {pos_weight.item():.1f}x (train: {n_neg} здоровых, {n_pos} меланом)")

Сплит 10/10: train 26500 (80.0%), val 3313 (10.0%), test 3313 (10.0%)
pos_weight = 55.6x (train: 26032 здоровых, 468 меланом)


In [23]:
# Базовая точка: лучшая аугментация + обычный батч + BCE (результат эксперимента 2)
BASELINE_METRICS = {
    "name":           "Baseline (лучшая аугм. + обычный батч + BCE)",
    "test_auc_pr":    0.2266,
    "test_f1":        0.2241,
    "test_recall":    0.4483,
    "test_precision": 0.1494,
    "test_spec":      0.9545,
    "test_bal_acc":   0.7014,
    "test_thr":       0.50,
}

In [24]:
# Загружаю ранее сохранённые результаты
strategy_results = load_results()

# Добавляю базовую точку, если её ещё нет
if not result_exists(strategy_results, BASELINE_METRICS["name"]):
    strategy_results.insert(0, BASELINE_METRICS)
    print(f"Базовая точка добавлена: {BASELINE_METRICS['name']}")

Файл результатов не найден.
Базовая точка добавлена: Baseline (лучшая аугм. + обычный батч + BCE)


In [28]:
# Стратегия 1: Оверсэмплинг
STRAT1 = "Оверсэмплинг"
if not result_exists(strategy_results, STRAT1):
    df_train_over = oversample_train(df_train_s)
    over_ds       = MelanomaDataset(df_train_over, IMAGE_DIR, transform=best_aug_transform)
    over_loader   = DataLoader(over_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=4, persistent_workers=True, pin_memory=True)
    result = run_experiment(STRAT1, over_loader, val_loader, test_loader)
    strategy_results.append(result)
    save_results(strategy_results)
else:
    print(f"'{STRAT1}' уже посчитан")

Оверсэмплинг: 26032 здоровых + 26032 меланом = 52064

--- Обучение: Оверсэмплинг ---
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 165MB/s] 


  Эп 01/10 | loss 0.2335 | val AUC-PR 0.0701 | F1 0.1357 <- лучший
  Эп 02/10 | loss 0.1007 | val AUC-PR 0.0976 | F1 0.2065 <- лучший
  Эп 03/10 | loss 0.0647 | val AUC-PR 0.0820 | F1 0.1549
  Эп 04/10 | loss 0.0523 | val AUC-PR 0.0951 | F1 0.1920
  Эп 05/10 | loss 0.0430 | val AUC-PR 0.0672 | F1 0.1429
  Эп 06/10 | loss 0.0358 | val AUC-PR 0.1007 | F1 0.1754 <- лучший
  Эп 07/10 | loss 0.0326 | val AUC-PR 0.1090 | F1 0.1714 <- лучший
  Эп 08/10 | loss 0.0251 | val AUC-PR 0.0759 | F1 0.1592
  Эп 09/10 | loss 0.0266 | val AUC-PR 0.1052 | F1 0.2143
  Эп 10/10 | loss 0.0241 | val AUC-PR 0.0977 | F1 0.1654
  TEST: AUC-PR 0.2106 | F1 0.3046 (thr=0.69) | Recall 0.3966 | Precision 0.2473 | Spec 0.9785 | BalAcc 0.6875


Сохранено в /kaggle/working/exp3_results.json


In [29]:
# Стратегия 2: Weighted BCE
STRAT2 = "Weighted BCE"
if not result_exists(strategy_results, STRAT2):
    result = run_experiment(
        STRAT2, regular_loader, val_loader, test_loader,
        criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    )
    strategy_results.append(result)
    save_results(strategy_results)
else:
    print(f"'{STRAT2}' уже посчитан")


--- Обучение: Weighted BCE ---
  Эп 01/10 | loss 1.0397 | val AUC-PR 0.0770 | F1 0.1595 <- лучший
  Эп 02/10 | loss 0.9697 | val AUC-PR 0.0863 | F1 0.1429 <- лучший
  Эп 03/10 | loss 0.9190 | val AUC-PR 0.0931 | F1 0.1739 <- лучший
  Эп 04/10 | loss 0.9008 | val AUC-PR 0.1394 | F1 0.1714 <- лучший
  Эп 05/10 | loss 0.8836 | val AUC-PR 0.0731 | F1 0.1290
  Эп 06/10 | loss 0.8559 | val AUC-PR 0.1179 | F1 0.1951
  Эп 07/10 | loss 0.8558 | val AUC-PR 0.1378 | F1 0.2017
  Эп 08/10 | loss 0.8221 | val AUC-PR 0.1241 | F1 0.1739
  Эп 09/10 | loss 0.8309 | val AUC-PR 0.1041 | F1 0.1915
  Эп 10/10 | loss 0.7978 | val AUC-PR 0.1611 | F1 0.2238 <- лучший
  TEST: AUC-PR 0.1579 | F1 0.2024 (thr=0.84) | Recall 0.2931 | Precision 0.1545 | Spec 0.9714 | BalAcc 0.6323


Сохранено в /kaggle/working/exp3_results.json


In [30]:
# Стратегия 3: Balanced batch
STRAT3 = "Balanced batch"
if not result_exists(strategy_results, STRAT3):
    result = run_experiment(STRAT3, balanced_loader, val_loader, test_loader)
    strategy_results.append(result)
    save_results(strategy_results)
else:
    print(f"'{STRAT3}' уже посчитан")


--- Обучение: Balanced batch ---
  Эп 01/10 | loss 0.2837 | val AUC-PR 0.0879 | F1 0.2032 <- лучший
  Эп 02/10 | loss 0.1469 | val AUC-PR 0.0892 | F1 0.1449 <- лучший
  Эп 03/10 | loss 0.1038 | val AUC-PR 0.1044 | F1 0.1970 <- лучший
  Эп 04/10 | loss 0.0801 | val AUC-PR 0.1057 | F1 0.1734 <- лучший
  Эп 05/10 | loss 0.0737 | val AUC-PR 0.1138 | F1 0.2379 <- лучший
  Эп 06/10 | loss 0.0565 | val AUC-PR 0.1053 | F1 0.1548
  Эп 07/10 | loss 0.0508 | val AUC-PR 0.0883 | F1 0.1766
  Эп 08/10 | loss 0.0439 | val AUC-PR 0.0717 | F1 0.1412
  Эп 09/10 | loss 0.0442 | val AUC-PR 0.0954 | F1 0.1978
  Эп 10/10 | loss 0.0375 | val AUC-PR 0.1178 | F1 0.2215 <- лучший
  TEST: AUC-PR 0.2548 | F1 0.2109 (thr=0.07) | Recall 0.5345 | Precision 0.1314 | Spec 0.9370 | BalAcc 0.7358


Сохранено в /kaggle/working/exp3_results.json


Focal Loss чувствителен к гиперпараметрам alpha и gamma - запускать с дефолтными значениями было бы нечестно по отношению к другим стратегиям. Использую Optuna для поиска лучшей комбинации гиперпараметров.

**Optuna TPE sampler** - байесовская оптимизация: учится на предыдущих trials и быстрее находит хорошую область, в отличие от случайного перебора.

**MedianPruner** - обрывает trial после PRUNER_WARMUP эпох, если его промежуточный результат хуже медианы всех остальных trials на той же эпохе. Это экономит GPU-время, не теряя качества поиска.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss решает проблему BCE при дисбалансе: умножает лосс каждого примера на (1-p)^gamma.
    Лёгкие примеры (высокая p) -> малый вес -> почти не учимся.
    Сложные (меланомы, малая p) -> вес близок к 1 -> активно учимся.

    alpha - дополнительный вес для меланом.
    gamma - сила подавления лёгких примеров (0 = обычный BCE).
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce          = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt           = torch.exp(-bce)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

In [25]:
def make_focal_objective(train_loader, val_loader):
    """Создаёт функцию для Optuna, которая:
      1. Получает от Optuna предложенные alpha и gamma
      2. Обучает модель с этими параметрами
      3. Возвращает val AUC-PR - Optuna запомнит результат и предложит
         лучшие параметры в следующей попытке
    Почему функция внутри функции?
    Optuna требует, чтобы её функция принимала только trial.
    Но ещё нужны train_loader и val_loader — они передаются снаружи, чтобы они были видны внутренней функции."""
    def objective(trial):
        # Optuna предлагает значения из заданных диапазонов
        alpha = trial.suggest_float("alpha", *ALPHA_RANGE)
        gamma = trial.suggest_float("gamma", *GAMMA_RANGE)
        criterion = FocalLoss(alpha=alpha, gamma=gamma)
        model     = make_resnet50()
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        best_val_auc_pr = 0.0

        for epoch in range(1, OPTUNA_EPOCHS + 1):
            model.train()
            for images, labels in train_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE).unsqueeze(1)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                # При экстремальных значениях гиперпараметров (например, alpha=0.49, gamma=0.66) Focal Loss вызывал взрыв градиентов: обучение прерывалось ошибкой Input contains NaN
                # Gradient clipping ограничивает норму градиентов до 1, направление обучения не меняется, только размер шага
                # Намеренно не использую функцию train_one_epoch, чтобы не модифицировать её для всех экспериментов, так как тогда будет нечестное сравнение остальных экспериментов
                # Gradient clipping нужен только при подборе гиперпараметров Focal Loss, где экстремальные значения alpha и gamma могут вызвать взрыв градиентов
                # Остальные стратегии были численно стабильны
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            val_metrics     = compute_metrics(model, val_loader, threshold=None)
            val_auc_pr      = val_metrics["auc_pr"]
            best_val_auc_pr = max(best_val_auc_pr, val_auc_pr)
            # Сообщаю Optuna о результате этой эпохи
            # Pruner сравнивает его с медианой других trials на той же эпохе
            trial.report(val_auc_pr, epoch)
            if trial.should_prune():
                # Trial явно хуже большинства - обрываю, не трачу GPU
                raise optuna.exceptions.TrialPruned()

        return best_val_auc_pr
    return objective


def run_optuna_focal(name, train_loader, val_loader, test_loader):
    """
    Запускает Optuna для подбора alpha/gamma, затем обучает финальную модель с лучшими параметрами и возвращает результат в стандартном формате.
    """
    print(f"Optuna: поиск гиперпараметров для '{name}' ({N_TRIALS} trials)...")

    # MedianPruner: не прунит первые N_STARTUP_TRIALS trials (слишком мало данных для медианы) и первые PRUNER_WARMUP эпох (модель ещё не разогрелась)
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=PRUNER_WARMUP
    )
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(
        make_focal_objective(train_loader, val_loader),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    best_alpha = study.best_params["alpha"]
    best_gamma = study.best_params["gamma"]
    print(f"  Лучшие параметры: alpha={best_alpha:.4f}, gamma={best_gamma:.4f}")
    print(f"  Лучший val AUC-PR в поиске: {study.best_value:.4f}")
    pruned  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"  Pruned trials: {pruned}/{N_TRIALS}")

    print(f"  Финальное обучение с alpha={best_alpha:.4f}, gamma={best_gamma:.4f}...")
    result = run_experiment(
        name, train_loader, val_loader, test_loader,
        criterion=FocalLoss(alpha=best_alpha, gamma=best_gamma)
    )

    result["optuna_alpha"] = best_alpha
    result["optuna_gamma"] = best_gamma
    return result

In [28]:
# Стратегия 4: Focal Loss
STRAT4 = "Focal Loss"
if not result_exists(strategy_results, STRAT4):
    result = run_optuna_focal(STRAT4, regular_loader, val_loader, test_loader)
    strategy_results.append(result)
    save_results(strategy_results)
else:
    print(f"'{STRAT4}' уже посчитан")
    r = get_result(strategy_results, STRAT4)
    alpha = r.get("optuna_alpha")
    gamma = r.get("optuna_gamma")
    if alpha is not None and gamma is not None:
        print(f"  alpha={alpha:.4f}, gamma={gamma:.4f}, AUC-PR={r['test_auc_pr']:.4f}")
    else:
        print(f"  AUC-PR={r['test_auc_pr']:.4f} (параметры Optuna не сохранены в файле)")

Optuna: поиск гиперпараметров для 'Focal Loss' (10 trials)...


  0%|          | 0/10 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 11%|█         | 10.6M/97.8M [00:00<00:00, 111MB/s]
 35%|███▍      | 33.9M/97.8M [00:00<00:00, 189MB/s]
 55%|█████▌    | 54.1M/97.8M [00:00<00:00, 199MB/s]
 76%|███████▋  | 74.8M/97.8M [00:00<00:00, 205MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 200MB/s]


  Лучшие параметры: alpha=0.2564, gamma=1.0102
  Лучший val AUC-PR в поиске: 0.1436
  Pruned trials: 1/10
  Финальное обучение с alpha=0.2564, gamma=1.0102...

--- Обучение: Focal Loss ---
  Эп 01/10 | loss 0.0107 | val AUC-PR 0.0943 | F1 0.1801 <- лучший
  Эп 02/10 | loss 0.0097 | val AUC-PR 0.0889 | F1 0.1676
  Эп 03/10 | loss 0.0095 | val AUC-PR 0.1068 | F1 0.1927 <- лучший
  Эп 04/10 | loss 0.0093 | val AUC-PR 0.1146 | F1 0.2093 <- лучший
  Эп 05/10 | loss 0.0092 | val AUC-PR 0.1525 | F1 0.2468 <- лучший
  Эп 06/10 | loss 0.0090 | val AUC-PR 0.1398 | F1 0.2212
  Эп 07/10 | loss 0.0089 | val AUC-PR 0.0957 | F1 0.1648
  Эп 08/10 | loss 0.0087 | val AUC-PR 0.1248 | F1 0.2126
  Эп 09/10 | loss 0.0085 | val AUC-PR 0.1568 | F1 0.2649 <- лучший
  Эп 10/10 | loss 0.0083 | val AUC-PR 0.0872 | F1 0.1707
  TEST: AUC-PR 0.2374 | F1 0.2639 (thr=0.25) | Recall 0.3276 | Precision 0.2209 | Spec 0.9794 | BalAcc 0.6535


Сохранено в /kaggle/working/exp3_results.json


In [26]:
# Стратегия 5: Balanced batch + Focal Loss
STRAT5 = "Balanced batch + Focal Loss"
if not result_exists(strategy_results, STRAT5):
    result = run_optuna_focal(STRAT5, balanced_loader, val_loader, test_loader)
    strategy_results.append(result)
    save_results(strategy_results)
else:
    print(f"'{STRAT5}' уже посчитан")
    r = get_result(strategy_results, STRAT5)
    alpha = r.get("optuna_alpha")
    gamma = r.get("optuna_gamma")
    if alpha is not None and gamma is not None:
        print(f"  alpha={alpha:.4f}, gamma={gamma:.4f}, AUC-PR={r['test_auc_pr']:.4f}")
    else:
        print(f"  AUC-PR={r['test_auc_pr']:.4f} (параметры Optuna не сохранены в файле)") 

Optuna: поиск гиперпараметров для 'Balanced batch + Focal Loss' (10 trials)...


  0%|          | 0/10 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 10%|▉         | 9.62M/97.8M [00:00<00:00, 101MB/s]
 28%|██▊       | 27.4M/97.8M [00:00<00:00, 151MB/s]
 46%|████▌     | 45.0M/97.8M [00:00<00:00, 166MB/s]
 64%|██████▍   | 62.8M/97.8M [00:00<00:00, 173MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


  Лучшие параметры: alpha=0.4296, gamma=2.7473
  Лучший val AUC-PR в поиске: 0.1800
  Pruned trials: 2/10
  Финальное обучение с alpha=0.4296, gamma=2.7473...

--- Обучение: Balanced batch + Focal Loss ---
  Эп 01/10 | loss 0.0216 | val AUC-PR 0.1172 | F1 0.1963 <- лучший
  Эп 02/10 | loss 0.0124 | val AUC-PR 0.0746 | F1 0.1365
  Эп 03/10 | loss 0.0092 | val AUC-PR 0.1172 | F1 0.2194 <- лучший
  Эп 04/10 | loss 0.0074 | val AUC-PR 0.1025 | F1 0.2148
  Эп 05/10 | loss 0.0061 | val AUC-PR 0.1025 | F1 0.1990
  Эп 06/10 | loss 0.0055 | val AUC-PR 0.1071 | F1 0.1709
  Эп 07/10 | loss 0.0043 | val AUC-PR 0.0908 | F1 0.1596
  Эп 08/10 | loss 0.0046 | val AUC-PR 0.0767 | F1 0.1688
  Эп 09/10 | loss 0.0049 | val AUC-PR 0.0942 | F1 0.1573
  Эп 10/10 | loss 0.0044 | val AUC-PR 0.0778 | F1 0.1544
  TEST: AUC-PR 0.1919 | F1 0.2659 (thr=0.67) | Recall 0.3966 | Precision 0.2000 | Spec 0.9717 | BalAcc 0.6841


Сохранено в /kaggle/working/exp3_results.json


In [28]:
# Загружаю каждый файл отдельно, так как обучение долгое, сессию приходилось обрывать каждые 12 часов
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

results_1 = load_json("/kaggle/input/datasets/nikitaliya/exp3-results/exp3_results (1).json")  # оверсэмплинг, weighted BCE, balanced batch
results_2 = load_json("/kaggle/input/datasets/nikitaliya/exp3-results/exp3_results (2).json")  # focal loss
results_3 = load_json("/kaggle/input/datasets/nikitaliya/res-exp3/exp3_results (3).json")      # focal loss + balanced batch

strategy_results = [BASELINE_METRICS] + results_1 + results_2 + results_3

# Дубли появились из-за того что BASELINE_METRICS добавлялся заново при каждой загрузке JSON
seen = set()
strategy_results_clean = []
for r in strategy_results:
    if r["name"] not in seen:
        seen.add(r["name"])
        strategy_results_clean.append(r)
strategy_results = strategy_results_clean

print("Загруженные стратегии:")
for r in strategy_results:
    print(f"  {r['name']}")
    print(
        f"    TEST: AUC-PR {r['test_auc_pr']:.4f} | "
        f"F1 {r['test_f1']:.4f} (thr={r['test_thr']:.2f}) | "
        f"Recall {r['test_recall']:.4f} | Precision {r['test_precision']:.4f} | "
        f"Spec {r['test_spec']:.4f} | BalAcc {r['test_bal_acc']:.4f}"
    )

Загруженные стратегии:
  Baseline (лучшая аугм. + обычный батч + BCE)
    TEST: AUC-PR 0.2266 | F1 0.2241 (thr=0.50) | Recall 0.4483 | Precision 0.1494 | Spec 0.9545 | BalAcc 0.7014
  Оверсэмплинг
    TEST: AUC-PR 0.2106 | F1 0.3046 (thr=0.69) | Recall 0.3966 | Precision 0.2473 | Spec 0.9785 | BalAcc 0.6875
  Weighted BCE
    TEST: AUC-PR 0.1579 | F1 0.2024 (thr=0.84) | Recall 0.2931 | Precision 0.1545 | Spec 0.9714 | BalAcc 0.6323
  Balanced batch
    TEST: AUC-PR 0.2548 | F1 0.2109 (thr=0.07) | Recall 0.5345 | Precision 0.1314 | Spec 0.9370 | BalAcc 0.7358
  Focal Loss
    TEST: AUC-PR 0.2374 | F1 0.2639 (thr=0.25) | Recall 0.3276 | Precision 0.2209 | Spec 0.9794 | BalAcc 0.6535
  Balanced batch + Focal Loss
    TEST: AUC-PR 0.1919 | F1 0.2659 (thr=0.67) | Recall 0.3966 | Precision 0.2000 | Spec 0.9717 | BalAcc 0.6841


## Вывод по Эксперименту 3

Лучшей стратегией борьбы с дисбалансом по основной метрике AUC-PR стал **Balanced batch (0,2548)**, что на 12,4% выше бейслайна (0,2266). Он также показал наилучший Recall (0,5345) и Balanced Accuracy (0,7358) - это означает, что модель научилась находить меланомы значительно лучше, сохраняя при этом приемлемое качество на здоровом классе.

Balanced batch выравнивает частоту подачи классов в модель во время обучения через WeightedRandomSampler, не изменяя данные физически. Это позволяет модели уделять равное внимание обоим классам на каждом батче, что особенно эффективно при соотношении классов 1:55.

**Focal Loss (AUC-PR 0,2374)** несмотря на более низкий AUC-PR, показал наилучший F1 (0,2639). Однако если рассматривать метрику Recall, говорящую о том, сколько меланом вообще было поймано, то видно, что этот показатель меньше на 20 процентных пунктов по сравнению с Balanced Batch, что соответственно отразилось и на Balanced Accuracy. 

**Оверсэмплинг (AUC-PR 0,2106)** показал результат ниже бейслайна, что объясняется риском переобучения: модель видит одни и те же 584 меланомы в среднем 50 раз за эпоху и запоминает конкретные примеры вместо обобщённых признаков.

**Balanced batch + Focal Loss (AUC-PR 0,1919)** - комбинация двух техник не дала синергии и показала результат хуже каждой из них по отдельности. Возможная причина: двойное давление на редкий класс (и через сэмплер, и через лосс) привело к нестабильности обучения

**Weighted BCE (AUC-PR 0,1579)** показал наихудший результат. Несмотря на математически обоснованный pos_weight = 55, штраф оказался слишком жёстким — модель нестабильно обучалась и не смогла найти хорошее разделение классов.

Стратегия **Balanced batch** переносится на не только на ResNet, но и на ViT и Swin Transformer как победитель по основной метрике AUC-PR.

In [28]:
# Сохраняю индексы сплита, чтобы все последующие ноутбуки работали ровно на тех же данных — это важно для честного сравнения архитектур
_, _, _, _, df_train_final = make_split_loaders(df, BEST_SIZE, train_transform=best_aug_transform)

Сплит 10/10: train 26500 (80.0%), val 3313 (10.0%), test 3313 (10.0%)


In [29]:
# Воссоздаю val и test сплиты, чтобы сохранить их image_name
df_tv, df_test = train_test_split(df, test_size=BEST_SIZE, random_state=RANDOM_SEED, stratify=df["target"])
df_train, df_val = train_test_split(df_tv, test_size=BEST_SIZE/(1-BEST_SIZE), random_state=RANDOM_SEED, stratify=df_tv["target"])

In [30]:
# Сохраняю только image_name и target — этого достаточно для воссоздания датасета
df_train[["image_name", "target"]].to_csv("/kaggle/working/split_train.csv", index=False)
df_val[["image_name", "target"]].to_csv("/kaggle/working/split_val.csv", index=False)
df_test[["image_name", "target"]].to_csv("/kaggle/working/split_test.csv", index=False)

In [31]:
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print("Сплит сохранён")

Train: 26500, Val: 3313, Test: 3313
Сплит сохранён
